# 1-4. Pandas 기반 데이터 품질 확인 실습

이 notebook의 목적은 모델 평가를 시작해도 되는 데이터 상태인지 판단하는 것입니다. 릴리스 회의 전에 `high_risk` 비율 변화가 보고되었지만, 아직 원인이 모델인지 데이터인지 알 수 없습니다. 수강생은 QA 담당자로서 데이터 구조, 입력값, 라벨 기준이 모델 지표 계산을 지탱하는지 먼저 확인해야 합니다.

이 실습은 생체신호 기반 위험 알림 시스템을 예제로 사용하지만 실제 의료 판단을 다루지 않습니다. `label`은 모델 평가용 정답 기준이고, Positive class는 `high_risk`, Negative class는 `low_risk`로 둡니다. 여기서 만드는 산출물은 2장의 모델 평가로 넘길 `chapter_01_evidence_packet`과 handoff 판단입니다.

| 받은 업무 | 현장 관측값 | 결정 압박 |
| --- | --- | --- |
| 평가 데이터가 모델 지표 계산에 적합한지 확인합니다 | `high_risk` 비율 변화가 보고되었지만 원인이 불명확합니다 | 17시 회의 전까지 모델 평가 진행, 조건부 진행, 데이터 재확인 중 하나를 제안해야 합니다 |

### 따라하기 안내: Pandas 데이터 품질 흐름

이 notebook은 Pandas를 처음 쓰는 사람도 따라갈 수 있게, 작은 예제로 기본 문법을 먼저 보고 실제 데이터 품질 확인으로 넘어갑니다.

1. 작은 DataFrame으로 Pandas 기본 조작을 익힙니다.
2. 실제 평가 데이터가 어디서 왔는지 확인합니다.
3. 필수 컬럼과 데이터 구조를 확인합니다.
4. 결측값과 범위 오류를 찾습니다.
5. label 값과 표본 수가 모델 평가에 충분한지 봅니다.
6. 2장 모델 평가로 넘길 판단 근거를 정리합니다.

출력 표를 볼 때는 “이 값이 데이터 구조, 결측, 범위, label 중 무엇을 확인하는 값인가?”를 계속 구분하세요.


## 0. Pandas를 처음 보는 사람을 위한 기초 예제

1장 실습은 Pandas로 표 데이터를 읽고 확인하는 과정입니다. 본 데이터에 들어가기 전에, 아주 작은 표로 Pandas의 기본 동작을 먼저 봅니다.

| Pandas 표현 | 쉬운 뜻 | 1장에서 쓰는 이유 |
| --- | --- | --- |
| `DataFrame` | 행과 컬럼이 있는 표 | CSV 데이터를 표로 다룹니다. |
| `head()` | 앞의 몇 행 보기 | 데이터 모양을 빠르게 확인합니다. |
| `shape` | 행 수와 컬럼 수 | 평가 분모와 구조를 확인합니다. |
| `isna()` | 값이 비어 있는지 확인 | 결측값을 찾습니다. |
| `value_counts()` | 값별 개수 세기 | label 분포를 봅니다. |
| 조건 필터 | 조건에 맞는 행만 보기 | 범위 오류를 찾습니다. |
| `assign()` | 새 컬럼을 붙인 표 만들기 | 품질 상태를 표에 추가합니다. |

이 예제는 Pandas 문법을 깊게 외우기 위한 것이 아닙니다. 뒤에서 나오는 데이터 품질 검사 셀이 어떤 일을 하는지 미리 손에 익히는 구간입니다.


### 0-1. 작은 DataFrame 만들기

아래 표에는 일부러 세 가지 문제가 들어 있습니다.

- `heart_rate`가 비어 있는 행
- `oxygen_saturation`이 100을 넘는 행
- `label`이 허용 목록에 없는 행

먼저 사람이 볼 수 있는 작은 표로 시작합니다.


In [1]:
import pandas as pd

pandas_basic_dataframe = pd.DataFrame(
    {
        "patient_id": [1, 2, 3, 4],
        "heart_rate": [82, None, 125, 74],
        "oxygen_saturation": [98, 95, 103, 97],
        "label": ["high_risk", "low_risk", "unknown", "low_risk"],
    }
)

display(pandas_basic_dataframe)


,patient_id,heart_rate,oxygen_saturation,label
0,1,82.0,98,high_risk
1,2,NaN,95,low_risk
2,3,125.0,103,unknown
3,4,74.0,97,low_risk


**출력 확인**

`DataFrame`은 행과 컬럼이 있는 표입니다. 지금은 4행짜리 작은 표라서 사람이 직접 문제를 찾을 수 있습니다.

하지만 실제 CSV는 행이 훨씬 많습니다. 그래서 Pandas로 결측, 범위 오류, label 분포를 계산합니다.


### 0-2. Pandas 기본 확인 함수 써 보기

이 셀은 뒤에서 반복해서 나오는 Pandas 기본 동작을 한 번에 보여 줍니다.

- `shape`: 행/컬럼 수
- `head()`: 앞부분 미리보기
- `isna().sum()`: 컬럼별 결측 개수
- `value_counts()`: label 값별 개수
- 조건 필터: 범위 오류 행 찾기


In [2]:
pandas_basic_shape = pd.DataFrame(
    [{"check": "shape", "rows": pandas_basic_dataframe.shape[0], "columns": pandas_basic_dataframe.shape[1]}]
)

pandas_basic_missing = (
    pandas_basic_dataframe
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

pandas_basic_label_counts = (
    pandas_basic_dataframe["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
)

pandas_basic_range_errors = pandas_basic_dataframe.loc[
    pandas_basic_dataframe["oxygen_saturation"].gt(100),
    ["patient_id", "oxygen_saturation", "label"],
]


**출력 확인: `pandas_basic_shape`**

먼저 표의 크기를 봅니다. 행 수는 검사 대상 개수이고, 컬럼 수는 확인할 정보의 종류입니다.


In [3]:
display(pandas_basic_shape)

,check,rows,columns
0,shape,4,4


**출력 확인: `head(2)`**

앞의 두 행만 보면서 컬럼 이름과 값 모양을 확인합니다. 이 단계는 전체 품질 판단이 아니라 데이터 모양 확인입니다.


In [4]:
display(pandas_basic_dataframe.head(2))

,patient_id,heart_rate,oxygen_saturation,label
0,1,82.0,98,high_risk
1,2,NaN,95,low_risk


**출력 확인: `pandas_basic_missing`**

컬럼별 결측 개수를 봅니다. `heart_rate`에 결측이 있으면 모델 입력값이 비어 있는 행이 있다는 뜻입니다.


In [5]:
display(pandas_basic_missing)

,column,missing_count
0,patient_id,0
1,heart_rate,1
2,oxygen_saturation,0
3,label,0


**출력 확인: `pandas_basic_label_counts`**

label 값별 개수를 봅니다. 허용 목록 밖의 값이 있으면 지표 계산 전에 label 기준을 확인해야 합니다.


In [6]:
display(pandas_basic_label_counts)

,label,count
0,low_risk,2
1,high_risk,1
2,unknown,1


**출력 확인: `pandas_basic_range_errors`**

조건 필터로 범위를 벗어난 행만 봅니다. 여기서는 `oxygen_saturation`이 100을 넘는 행이 표시됩니다.


In [7]:
display(pandas_basic_range_errors)

,patient_id,oxygen_saturation,label
2,3,103,unknown


**0-2 정리**

방금 본 출력들은 뒤 실습의 기본 재료입니다. 실제 데이터에서도 같은 방식으로 행/컬럼 수, 미리보기, 결측, label 분포, 범위 오류를 차례대로 확인합니다.


### 0-3. 품질 상태 컬럼 붙이기

`assign()`은 기존 표를 바탕으로 새 컬럼을 붙인 표를 만들 때 자주 씁니다. 여기서는 행마다 품질 상태를 쉽게 표시해 봅니다.


In [8]:
pandas_basic_flagged = pandas_basic_dataframe.assign(
    heart_rate_missing=lambda table: table["heart_rate"].isna(),
    oxygen_out_of_range=lambda table: table["oxygen_saturation"].gt(100),
    label_allowed=lambda table: table["label"].isin(["high_risk", "low_risk"]),
)

display(pandas_basic_flagged)


,patient_id,heart_rate,oxygen_saturation,label,heart_rate_missing,oxygen_out_of_range,label_allowed
0,1,82.0,98,high_risk,False,False,True
1,2,NaN,95,low_risk,True,False,True
2,3,125.0,103,unknown,False,True,False
3,4,74.0,97,low_risk,False,False,True


**출력 확인**

새로 붙은 세 컬럼은 각 행의 품질 상태를 `True` 또는 `False`로 보여 줍니다.

이제 1장의 실제 실습으로 넘어가면 같은 생각을 더 큰 데이터에 적용합니다. 먼저 데이터 출처와 컬럼 구조를 보고, 결측과 범위 오류, label 기준을 차례대로 확인합니다.


## 1. 실습 흐름과 도구 역할

### 1-1. 이 notebook에서 판단할 질문

이 notebook은 Pandas 사용법을 외우는 자료가 아니라 데이터 품질 판단을 보고서 문장으로 바꾸는 실습 교재입니다. 각 단계는 먼저 판단 질문을 제시하고, 코드 셀은 그 판단에 필요한 evidence table을 만듭니다. 출력값은 정답 확인용이 아니라 “모델 평가로 넘어가도 되는가”를 설명하는 근거입니다.

`high_risk` 비율 변화가 보이면 모델을 바로 의심하기 쉽습니다. 하지만 잘못된 CSV를 읽었거나, 라벨 기준이 흔들렸거나, 특정 입력 특성에 결측 또는 범위 오류가 섞였으면 모델 지표는 원인 판단을 흐릴 수 있습니다. 따라서 1장에서는 데이터 출처, schema, 결측, 허용 범위, 라벨 support를 순서대로 확인합니다.

| 단계 | gate | 통과하면 가능한 판단 | 실패하면 필요한 action |
| --- | --- | --- | --- |
| 1 | 데이터 출처 고정 | 같은 파일을 기준으로 이후 검사를 진행할 수 있습니다 | 파일 경로와 생성 절차를 확인합니다 |
| 2 | schema gate | 필수 입력과 라벨 구조가 준비되었습니다 | 데이터 추출 또는 컬럼명 정리 owner 확인 |
| 3 | missing gate | 결측으로 인한 평가 전제 훼손 가능성이 낮습니다 | 결측 위치와 라벨 집중도를 확인합니다 |
| 4 | range gate | 명백한 입력 범위 오류 가능성이 낮습니다 | 수집 단위, 파싱, 오류 코드 유입을 확인합니다 |
| 5 | label gate | 모델 지표 해석에 필요한 정답 기준이 준비되었습니다 | 라벨 생성 기준과 표본 수를 재확인합니다 |
| 6 | handoff | 2장 모델 평가로 넘길 근거가 정리됩니다 | 보류 조건, owner, next action을 남깁니다 |

### 1-2. `ai-quality` 패키지와 `utils.py`의 책임

이 notebook은 공통 기준값과 브라우저 실행 helper를 `utils.py`를 통해 준비합니다. JupyterLite에서는 `utils.py`가 `ttamlops-ai-quality-lite` wheel을 설치하고, 로컬 일반 Jupyter에서는 저장소의 `packages/ai-quality/src`를 우선 사용합니다.

비핵심 로직은 `utils.py`로 분리했습니다. Notebook 셀에는 실습자가 읽어야 하는 판단 흐름과 핵심 실행 호출만 남깁니다. `utils.py`는 패키지 준비, `ai_quality.lite` import, 1장 전용 gate table과 evidence packet 조립을 맡습니다.

| 구성 | 책임 |
| --- | --- |
| `ai_quality.lite` | Lite에서도 쓸 수 있는 공통 라벨, 컬럼, 범위, 데이터 로딩 helper 제공 |
| `labs/ch01_data_quality/utils.py` | Lite/package 준비, 1장 workbook 전용 gate table과 evidence packet 조립 |
| 이 notebook | 업무 배경, 실행 순서, 출력 해석, 보고 문장 작성 |

### 따라하기 안내: 환경 준비

**이 셀에서 하는 일**  
실습에 필요한 도구를 불러옵니다.

**확인할 점**  
패키지 이름, 기준 label, threshold처럼 뒤 셀에서 계속 사용할 기준값을 확인합니다.

**왜 중요한가**  
처음 기준이 잘못 잡히면 뒤의 모든 출력도 잘못 해석됩니다.


In [9]:
from __future__ import annotations

import utils as ch01

prepared = await ch01.prepare_notebook()
pd = prepared.pandas

environment_summary = pd.DataFrame(
    [
        {"item": "package_module", "value": ch01.aiq_lite.__name__, "why_it_matters": "데이터 기준과 label 기준을 제공하는 모듈입니다."},
        {"item": "utils_module", "value": ch01.__name__, "why_it_matters": "notebook 실행 환경과 파일 경로를 준비합니다."},
    ]
)

display(environment_summary)


,item,value,why_it_matters
0,package_module,ai_quality.lite,데이터 기준과 label 기준을 제공하는 모듈입니다.
1,utils_module,utils,notebook 실행 환경과 파일 경로를 준비합니다.


## 2. 실습 데이터 이해

### 2-1. 기준 데이터 로딩과 원본 관찰

1장의 첫 판단은 이 notebook이 어떤 데이터를 근거로 쓰는지 확인하는 것입니다. 같은 코드라도 로컬 전체 데이터와 JupyterLite sample은 행 수가 다를 수 있으므로, 수강생은 숫자를 보기 전에 `data_source`, 행 수, 컬럼 수, 원본 라벨 값을 먼저 고정해야 합니다.

이 셀에서는 데이터 로딩을 `utils.py` 뒤에 숨기지 않고 notebook에서 직접 실행합니다. `ai_quality.lite`의 browser-safe loader로 CSV 또는 fallback sample을 읽고, Pandas의 `shape`, `head()`, `value_counts()`, `assign()`으로 원본 상태를 먼저 관찰합니다. 아직 이 단계에서는 `label` 표준화를 적용하지 않습니다.


### 따라하기 안내: feature와 원본 데이터 확인

**이 셀에서 하는 일**  
원본 데이터와 모델 입력 feature 목록을 확인합니다.

**확인할 점**  
행 수, 컬럼 수, 주요 feature, 원본 label 값을 봅니다.

**왜 중요한가**  
아직 품질 결론을 내리는 단계가 아닙니다. 어떤 데이터에서 시작하는지만 고정합니다.


In [10]:
feature_columns = ch01.aiq_lite.FEATURE_COLUMNS
required_columns = ch01.aiq_lite.REQUIRED_COLUMNS

raw_dataframe, data_source = ch01.aiq_lite.load_csv_or_sample(
    ch01.DATA_PATH,
    ch01.aiq_lite.sample_vital_signs(),
    nrows=None,
)

raw_label_distribution = (
    raw_dataframe["label"]
    .value_counts(dropna=False)
    .rename_axis("raw_label")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(raw_dataframe) * 100).round(2))
)

raw_snapshot = pd.DataFrame(
    [
        {"check": "data_source", "observed": data_source},
        {"check": "row_count", "observed": raw_dataframe.shape[0]},
        {"check": "column_count", "observed": raw_dataframe.shape[1]},
        {
            "check": "feature_columns_present",
            "observed": f"{len(set(feature_columns) & set(raw_dataframe.columns))}/{len(feature_columns)}",
        },
    ]
).assign(
    qa_use=[
        "파일 또는 fallback sample 기준을 고정합니다.",
        "모델 평가 전제 확인의 분모로 기록합니다.",
        "실습 schema와 실제 컬럼 수를 비교합니다.",
        "모델 입력 feature 누락 여부를 schema gate 전에 확인합니다.",
    ]
)


### 출력 확인: `raw_snapshot`

**확인할 점**  
행 수, 컬럼 수, 데이터셋 이름을 봅니다.

**왜 중요한가**  
분석 대상이 기대와 다르면 뒤의 결측, label, metric 결과도 의미가 달라집니다.

**다음 단계**  
대상이 맞으면 preview와 label 분포로 넘어갑니다.


In [11]:
display(raw_snapshot)


,check,observed,qa_use
0,data_source,embedded JupyterLite sample,파일 또는 fallback sample 기준을 고정합니다.
1,row_count,12,모델 평가 전제 확인의 분모로 기록합니다.
2,column_count,10,실습 schema와 실제 컬럼 수를 비교합니다.
3,feature_columns_present,6/6,모델 입력 feature 누락 여부를 schema gate 전에 확인합니다.


### 출력 확인: `raw_dataframe.head()`

**확인할 점**  
몇 개 행을 직접 보고 컬럼 이름과 값 모양이 예상과 맞는지 봅니다.

**왜 중요한가**  
Preview는 전체 판단이 아니라 데이터 모양 확인입니다.

**다음 단계**  
분포나 metric은 뒤의 요약 표에서 확인합니다.


In [12]:
display(raw_dataframe.head())


,patient_id,timestamp,heart_rate,respiratory_rate,body_temperature,oxygen_saturation,systolic_blood_pressure,diastolic_blood_pressure,age,label
0,p-001,2026-01-01T09:00:00Z,72.0,16,36.6,98,118,76,38,low_risk
1,p-002,2026-01-01T09:01:00Z,118.0,25,37.9,92,146,92,65,high_risk
2,p-003,2026-01-01T09:02:00Z,63.0,15,36.4,99,110,70,29,low_risk
3,p-004,2026-01-01T09:03:00Z,132.0,28,38.4,89,158,96,72,high_risk
4,p-005,2026-01-01T09:04:00Z,84.0,18,36.8,97,126,78,41,low_risk


### 출력 확인: `raw_label_distribution`

**확인할 점**  
`high_risk`와 `low_risk`의 개수와 비율을 봅니다.

**왜 중요한가**  
`high_risk`는 관심 class입니다. 이 수가 적으면 recall 같은 지표가 크게 흔들릴 수 있습니다.

**다음 단계**  
두 class가 모두 있는지 확인한 뒤 label gate로 넘어갑니다.


In [13]:
display(raw_label_distribution)


,raw_label,count,ratio_pct
0,low_risk,6,50.0
1,high_risk,6,50.0


이 출력에서 확인할 핵심은 원본 데이터가 어디서 왔고, 표준화 전 `label`이 어떤 값으로 들어왔는지입니다. 로컬 전체 데이터라면 더 큰 행 수가 보이고, Lite에서는 정적 사이트에 포함된 sample 규모가 보일 수 있습니다. 두 경우 모두 같은 gate 구조를 적용하지만, 보고서에는 실행 범위를 함께 적어야 합니다.

원본 라벨 값이 이미 `high_risk`, `low_risk`일 수도 있고, 다른 표기에서 표준화되어야 할 수도 있습니다. 따라서 다음 셀에서는 원본 dataframe을 직접 수정하지 않고 `copy()`와 `assign()`으로 평가용 dataframe을 별도로 만듭니다.

### 2-2. `label` 표준화와 context 구성

1장에서 gate에 넣는 dataframe은 원본을 그대로 쓰지 않고, 라벨 표기를 수업 기준으로 표준화한 평가용 dataframe입니다. 이 변환을 셀로 드러내야 수강생이 “원본 데이터”, “표준화된 label”, “모델 평가 전제”를 구분할 수 있습니다.

이 셀에서는 `DataFrame.copy()`, `assign()`, `map()`, `value_counts()`, `merge()`로 라벨 변환 과정을 확인합니다. 마지막에는 뒤쪽 gate와 evidence packet이 사용할 `context`를 구성하지만, context는 숨겨진 로딩 함수가 아니라 방금 만든 dataframe과 provenance를 묶는 컨테이너로만 사용합니다.


### 따라하기 안내: label 표준화

**이 셀에서 하는 일**  
label 표기를 `high_risk`, `low_risk`로 맞춥니다.

**확인할 점**  
`High Risk`, `high_risk`, `HIGH_RISK`처럼 같은 뜻의 값이 섞여 있지 않은지 봅니다.

**왜 중요한가**  
label은 모델 평가의 정답입니다. 표기가 섞이면 정답 기준이 흔들려 precision과 recall이 잘못 계산될 수 있습니다.


In [14]:
dataframe = raw_dataframe.copy()
if "label" in dataframe.columns:
    dataframe = dataframe.assign(
        label=lambda table: table["label"].map(ch01.aiq_lite.normalize_label)
    )

execution_scope = "lite_sample_or_local_file"
if "embedded" in data_source:
    execution_scope = "embedded_fallback_sample"
elif len(dataframe) <= 600:
    execution_scope = "jupyterlite_sample"
else:
    execution_scope = "local_full_or_large_sample"

label_transform = (
    pd.DataFrame(
        {
            "raw_label": raw_dataframe["label"],
            "standardized_label": dataframe["label"],
        }
    )
    .value_counts(["raw_label", "standardized_label"], dropna=False)
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(dataframe) * 100).round(2))
)

preview_columns = [
    column
    for column in ["patient_id", "timestamp", *feature_columns, "label"]
    if column in dataframe.columns
]
label_distribution = (
    dataframe["label"]
    .value_counts(dropna=False)
    .rename_axis("label")
    .reset_index(name="count")
    .assign(ratio_pct=lambda table: (table["count"] / len(dataframe) * 100).round(2))
)
provenance = pd.DataFrame(
    [
        {
            "package_module": ch01.aiq_lite.__name__,
            "data_path": ch01.DATA_PATH,
            "data_source": data_source,
            "execution_scope": execution_scope,
            "row_count": dataframe.shape[0],
            "column_count": dataframe.shape[1],
        }
    ]
)

context = ch01.ChapterOneContext(
    dataframe=dataframe,
    data_source=data_source,
    execution_scope=execution_scope,
    provenance=provenance,
    preview=dataframe.loc[:, preview_columns].head(5),
    label_distribution=label_distribution,
)


### 출력 확인: `label_transform`

**확인할 점**  
원본 label이 표준화 후 어떤 값으로 바뀌었는지 봅니다.

**왜 중요한가**  
label은 모델 평가의 정답입니다. 같은 뜻의 값이 여러 표기로 섞이면 정답 기준이 흔들립니다.

**다음 단계**  
표준화 결과가 `high_risk`, `low_risk`로 정리됐는지 확인합니다.

**주의할 점**  
표준화 결과만 보지 말고 원본 값도 같이 봅니다.


In [15]:
display(label_transform)


,raw_label,standardized_label,count,ratio_pct
0,high_risk,high_risk,6,50.0
1,low_risk,low_risk,6,50.0


### 출력 확인: `context.provenance`

**확인할 점**  
데이터나 artifact가 어디에서 왔는지 봅니다. 경로와 실행 범위를 먼저 확인합니다.

**왜 중요한가**  
같은 숫자라도 prepared artifact인지 local 재생성인지에 따라 보고서 표현이 달라집니다.

**다음 단계**  
이 범위를 기억한 상태로 다음 표를 읽습니다.


In [16]:
display(context.provenance)


,package_module,data_path,data_source,execution_scope,row_count,column_count
0,ai_quality.lite,data/vital_signs_evaluation_baseline.csv,embedded JupyterLite sample,embedded_fallback_sample,12,10


### 출력 확인: `context.preview`

**확인할 점**  
몇 개 행을 직접 보고 컬럼 이름과 값 모양이 예상과 맞는지 봅니다.

**왜 중요한가**  
Preview는 전체 판단이 아니라 데이터 모양 확인입니다.

**다음 단계**  
분포나 metric은 뒤의 요약 표에서 확인합니다.


In [17]:
display(context.preview)


,patient_id,timestamp,heart_rate,respiratory_rate,body_temperature,oxygen_saturation,systolic_blood_pressure,diastolic_blood_pressure,label
0,p-001,2026-01-01T09:00:00Z,72.0,16,36.6,98,118,76,low_risk
1,p-002,2026-01-01T09:01:00Z,118.0,25,37.9,92,146,92,high_risk
2,p-003,2026-01-01T09:02:00Z,63.0,15,36.4,99,110,70,low_risk
3,p-004,2026-01-01T09:03:00Z,132.0,28,38.4,89,158,96,high_risk
4,p-005,2026-01-01T09:04:00Z,84.0,18,36.8,97,126,78,low_risk


### 출력 확인: `context.label_distribution`

**확인할 점**  
`high_risk`와 `low_risk`의 개수와 비율을 봅니다.

**왜 중요한가**  
`high_risk`는 관심 class입니다. 이 수가 적으면 recall 같은 지표가 크게 흔들릴 수 있습니다.

**다음 단계**  
두 class가 모두 있는지 확인한 뒤 label gate로 넘어갑니다.


In [18]:
display(context.label_distribution)


,label,count,ratio_pct
0,low_risk,6,50.0
1,high_risk,6,50.0


이 출력에서 확인할 핵심은 `raw_label`이 어떤 `standardized_label`로 바뀌었는지입니다. 표준화 후 `high_risk`와 `low_risk`가 모두 관측되면 다음 단계에서 label gate를 수행할 수 있습니다. 한쪽 라벨이 없거나 예상하지 못한 라벨이 섞이면 모델 지표 계산보다 라벨 생성 기준 확인을 먼저 요청해야 합니다.

`execution_scope`는 보고서의 제한 사항입니다. Lite sample로 실행했다면 수치가 로컬 전체 데이터와 다를 수 있으므로, report sentence에는 sample 기준이라는 조건을 남겨야 합니다.

### 2-3. `label`과 모델 산출물의 경계 확인

1장에서 데이터에 들어 있어야 하는 값은 `feature`와 `label`입니다. `score`, `threshold`, `prediction`, `metric`은 원본 데이터에 있으면 오히려 해석을 의심해야 하는 값입니다. 이 값들은 2장 모델 평가나 3장 이후 서빙/운영 관측에서 생성됩니다.

이 셀에서는 `DataFrame.columns`를 기준으로 개념별 후보 컬럼이 실제 데이터에 있는지 확인합니다. `map()`, `assign()`, `apply()`를 사용해 후보 컬럼 목록을 관측 컬럼 목록으로 바꾸고, 1장 데이터에 있어야 하는 항목과 없어야 하는 항목을 비교합니다.


### 따라하기 안내: 품질 기준표 확인

**이 셀에서 하는 일**  
뒤에서 쓸 품질 기준을 먼저 확인합니다.

**확인할 점**  
허용 label, 수치 범위, 최소 표본 수 같은 기준을 봅니다.

**왜 중요한가**  
기준을 모르면 pass/fail 결과가 왜 나왔는지 설명할 수 없습니다.


In [19]:
concept_rules = pd.DataFrame(
    [
        {
            "concept": "feature",
            "candidate_columns": feature_columns,
            "expected_in_chapter_1_data": True,
            "qa_meaning": "1장에서 결측과 범위를 확인합니다.",
        },
        {
            "concept": "label",
            "candidate_columns": ["label"],
            "expected_in_chapter_1_data": True,
            "qa_meaning": "1장에서 허용 라벨과 class support를 확인합니다.",
        },
        {
            "concept": "score",
            "candidate_columns": ["score", "risk_score"],
            "expected_in_chapter_1_data": False,
            "qa_meaning": "2장 모델 평가 또는 운영 관측에서 생성합니다.",
        },
        {
            "concept": "threshold",
            "candidate_columns": ["threshold"],
            "expected_in_chapter_1_data": False,
            "qa_meaning": "2장에서 score를 class로 바꾸는 기준입니다.",
        },
        {
            "concept": "prediction",
            "candidate_columns": ["prediction"],
            "expected_in_chapter_1_data": False,
            "qa_meaning": "2장 또는 3장 이후 생성된 모델 출력입니다.",
        },
        {
            "concept": "metric",
            "candidate_columns": ["metric", "accuracy", "precision", "recall"],
            "expected_in_chapter_1_data": False,
            "qa_meaning": "label과 prediction 비교 뒤 계산합니다.",
        },
    ]
)

concept_presence = (
    concept_rules.assign(
        observed_columns=lambda table: table["candidate_columns"].map(
            lambda columns: [column for column in columns if column in dataframe.columns]
        )
    )
    .assign(
        observed_count=lambda table: table["observed_columns"].map(len),
        expected_count=lambda table: table["candidate_columns"].map(len),
        status=lambda table: table.apply(
            lambda row: "as_expected"
            if (
                row["observed_count"] == row["expected_count"]
                if row["expected_in_chapter_1_data"]
                else row["observed_count"] == 0
            )
            else "review",
            axis=1,
        ),
        candidate_columns=lambda table: table["candidate_columns"].map(
            lambda columns: ", ".join(columns)
        ),
        observed_columns=lambda table: table["observed_columns"].map(
            lambda columns: ", ".join(columns) if columns else "none"
        ),
    )
)

display(concept_presence)


,concept,candidate_columns,expected_in_chapter_1_data,qa_meaning,observed_columns,observed_count,expected_count,status
0,feature,"heart_rate, respiratory_rate, body_temperature...",True,1장에서 결측과 범위를 확인합니다.,"heart_rate, respiratory_rate, body_temperature...",6,6,as_expected
1,label,label,True,1장에서 허용 라벨과 class support를 확인합니다.,label,1,1,as_expected
2,score,"score, risk_score",False,2장 모델 평가 또는 운영 관측에서 생성합니다.,none,0,2,as_expected
3,threshold,threshold,False,2장에서 score를 class로 바꾸는 기준입니다.,none,0,1,as_expected
4,prediction,prediction,False,2장 또는 3장 이후 생성된 모델 출력입니다.,none,0,1,as_expected
5,metric,"metric, accuracy, precision, recall",False,label과 prediction 비교 뒤 계산합니다.,none,0,4,as_expected


`feature`와 `label`만 `as_expected`로 존재하고, `score`, `threshold`, `prediction`, `metric`은 존재하지 않는 상태가 1장 기준으로 자연스럽습니다. 만약 `prediction`이나 `risk_score`가 이 기준 데이터에 섞여 있으면, 평가 전 데이터와 모델 산출물이 같은 파일에 섞였는지 확인해야 합니다.

이 구분은 보고서 문장을 제한합니다. 1장에서는 “라벨 기준과 입력 feature 조건을 확인했다”고 쓸 수 있지만, “모델이 `high_risk`를 많이 예측했다”고 쓰면 안 됩니다. 예측 분포는 뒤 장의 모델 평가와 운영 로그에서 확인합니다.

### 2-4. 컬럼 역할과 실제 관측값 연결

데이터 파일의 모든 컬럼이 모델 입력은 아닙니다. 모델 입력 feature, 정답 기준 label, 추적용 metadata, 품질 참고 컬럼, 파생 참고 컬럼을 구분해야 schema gate와 missing gate의 의미가 분명해집니다.

이 셀에서는 `DataFrame.dtypes`, `isna().sum()`, `apply()`, `merge()`, `assign()`으로 컬럼별 관측 지도를 직접 만듭니다. 역할 분류는 수업 기준값을 쓰지만, dtype, 결측 수, sample value, range rule은 현재 dataframe에서 계산합니다.


### 따라하기 안내: 참고 컬럼 확인

**이 셀에서 하는 일**  
모델 입력이 아닌 참고 컬럼을 확인합니다.

**확인할 점**  
추적에 쓰는 id, 시간, 키/몸무게 같은 참고값이 있는지 봅니다.

**왜 중요한가**  
모든 컬럼이 모델 입력은 아닙니다. feature, label, metadata를 구분해야 합니다.


In [20]:
quality_reference_columns = ["age", "gender", "weight_kg", "height_m"]
derived_reference_columns = [
    "derived_hrv",
    "derived_pulse_pressure",
    "derived_bmi",
    "derived_map",
]
known_columns = [
    "patient_id",
    *feature_columns,
    "timestamp",
    *quality_reference_columns,
    *derived_reference_columns,
    "label",
]
ordered_columns = [column for column in known_columns if column in dataframe.columns]
ordered_columns.extend(
    [column for column in dataframe.columns if column not in ordered_columns]
)

def column_role(column: str) -> str:
    if column in feature_columns:
        return "model_input_feature"
    if column == "label":
        return "label"
    if column in {"patient_id", "timestamp"}:
        return "traceability_metadata"
    if column in quality_reference_columns:
        return "quality_reference"
    if column in derived_reference_columns:
        return "derived_reference"
    return "unexpected_or_extra"

valid_range_table = (
    pd.DataFrame.from_dict(
        ch01.aiq_lite.VALID_RANGES,
        orient="index",
        columns=["valid_min", "valid_max"],
    )
    .rename_axis("column")
    .reset_index()
    .assign(range_rule=lambda table: table["valid_min"].astype(str) + "~" + table["valid_max"].astype(str))
)
range_rules = valid_range_table.loc[:, ["column", "range_rule"]]

dtype_table = (
    dataframe.dtypes.astype(str)
    .rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_table = (
    dataframe.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
sample_table = (
    dataframe.apply(lambda series: series.dropna().iloc[0] if series.dropna().size else "all_missing")
    .astype(str)
    .rename("sample_value")
    .reset_index()
    .rename(columns={"index": "column"})
)

column_observation = (
    pd.DataFrame({"column": ordered_columns})
    .assign(role=lambda table: table["column"].map(column_role))
    .merge(dtype_table, on="column", how="left")
    .merge(missing_table, on="column", how="left")
    .merge(sample_table, on="column", how="left")
    .merge(range_rules, on="column", how="left")
    .assign(
        missing_ratio_pct=lambda table: (table["missing_count"] / len(dataframe) * 100).round(2),
        range_rule=lambda table: table["range_rule"].fillna("not_applicable"),
    )
)

display(column_observation)


,column,role,dtype,missing_count,sample_value,range_rule,missing_ratio_pct
0,patient_id,traceability_metadata,object,0,p-001,not_applicable,0.00
1,heart_rate,model_input_feature,float64,1,72.0,1.0~250.0,8.33
2,respiratory_rate,model_input_feature,int64,0,16,1.0~80.0,0.00
3,body_temperature,model_input_feature,float64,0,36.6,30.0~45.0,0.00
4,oxygen_saturation,model_input_feature,int64,0,98,0.0~100.0,0.00
5,systolic_blood_pressure,model_input_feature,int64,0,118,50.0~250.0,0.00
6,diastolic_blood_pressure,model_input_feature,int64,0,76,30.0~150.0,0.00
7,timestamp,traceability_metadata,object,0,2026-01-01T09:00:00Z,not_applicable,0.00
8,age,quality_reference,int64,0,38,0.0~120.0,0.00
9,label,label,object,0,low_risk,not_applicable,0.00


컬럼 역할 지도에서 우선 볼 대상은 `model_input_feature`와 `label`입니다. 모델 입력 feature의 결측이나 범위 문제는 score 계산과 metric 해석을 제한하고, `label` 문제는 Precision/Recall 계산 자체를 흔듭니다.

`patient_id`와 `timestamp`는 원인 추적에는 필요하지만 현재 모델 입력으로 해석하지 않습니다. `age`, `gender`, `weight_kg`, `height_m`, `derived_*` 컬럼도 데이터 품질 참고에는 쓸 수 있으나, 현재 모델 지표의 입력 근거로 과장하면 안 됩니다.

### 2-5. Feature 값의 관측 범위 확인

Range gate로 넘어가기 전에 모델 입력 feature의 실제 최솟값과 최댓값을 먼저 관찰해야 합니다. 허용 범위 표만 외우면 gate가 형식적인 검사처럼 보이지만, 실제 관측 범위를 보면 어떤 입력이 metric 해석을 흔들 수 있는지 더 분명해집니다.

이 셀에서는 `loc`로 모델 입력 feature만 선택하고, `apply(pd.to_numeric)`, `agg()`, `isna().sum()`, `lt()`, `gt()`로 관측 범위와 범위 초과 후보를 계산합니다. 뒤의 formal range gate는 이 사전 관측을 더 엄격한 QA gate로 바꾸는 단계입니다.


### 따라하기 안내: 수치 feature 요약

**이 셀에서 하는 일**  
수치 feature의 기본 통계를 확인합니다.

**확인할 점**  
평균, 최소, 최대를 보고 말이 안 되는 값이 있는지 봅니다.

**왜 중요한가**  
값의 범위가 이상하면 모델 score도 흔들릴 수 있습니다.


In [21]:
feature_values = dataframe.loc[:, feature_columns].apply(pd.to_numeric, errors="coerce")
feature_ranges = valid_range_table.set_index("column").loc[feature_columns]

feature_min_max = (
    feature_values.agg(["min", "max"])
    .T
    .rename(columns={"min": "observed_min", "max": "observed_max"})
    .rename_axis("feature")
    .reset_index()
)
feature_missing = (
    feature_values.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "feature"})
)
out_of_range_count = (
    feature_values.lt(feature_ranges["valid_min"], axis="columns")
    | feature_values.gt(feature_ranges["valid_max"], axis="columns")
).sum()
feature_out_of_range = (
    out_of_range_count.rename("out_of_range_count")
    .reset_index()
    .rename(columns={"index": "feature"})
)

feature_profile = (
    feature_min_max
    .merge(feature_missing, on="feature", how="left")
    .merge(feature_out_of_range, on="feature", how="left")
    .merge(
        feature_ranges.reset_index().rename(columns={"column": "feature"}),
        on="feature",
        how="left",
    )
    .assign(
        observed_min=lambda table: table["observed_min"].round(4),
        observed_max=lambda table: table["observed_max"].round(4),
    )
)

display(feature_profile)


,feature,observed_min,observed_max,missing_count,out_of_range_count,valid_min,valid_max,range_rule
0,heart_rate,63.0,141.0,1,0,1.0,250.0,1.0~250.0
1,respiratory_rate,14.0,31.0,0,0,1.0,80.0,1.0~80.0
2,body_temperature,36.2,38.8,0,0,30.0,45.0,30.0~45.0
3,oxygen_saturation,86.0,101.0,0,1,0.0,100.0,0.0~100.0
4,systolic_blood_pressure,110.0,164.0,0,0,50.0,250.0,50.0~250.0
5,diastolic_blood_pressure,70.0,102.0,0,0,30.0,150.0,30.0~150.0


Feature 관측 범위에서 `missing_count`와 `out_of_range_count`가 0이면 명백한 입력 결측과 범위 오류 가능성은 약해집니다. 다만 이 말은 운영 분포 변화가 없다는 뜻이 아닙니다. 운영 current batch 변화는 4장과 5장의 로그와 분포 근거로 다시 확인해야 합니다.

이제 데이터 로딩, 라벨 표준화, context 구성, label 경계, 컬럼 역할, feature 관측 범위를 확인했습니다. 다음 섹션부터는 같은 dataframe에 대해 schema, missing, range, label gate를 순서대로 적용하고, 각 출력이 2장 모델 평가로 넘길 evidence packet에 어떻게 들어가는지 확인합니다.


## 3. 데이터 출처와 라벨 기준 고정

### 3-1. 데이터 provenance 확인

첫 번째 판단은 내가 기대한 평가 데이터를 읽고 있는지 확인하는 것입니다. 잘못된 파일이나 fallback sample을 읽고도 이후 숫자만 보고하면, 모델 문제와 데이터 경로 문제를 구분할 수 없습니다. 따라서 데이터 출처와 실행 범위는 metric보다 먼저 보고서에 남겨야 합니다.

JupyterLite에서는 정적 사이트에 포함된 소형 CSV를 읽습니다. 로컬 일반 Jupyter에서는 저장소의 전체 또는 더 큰 CSV를 읽을 수 있습니다. 두 경로 모두 실습 판단은 가능하지만, 보고서에는 `execution_scope`를 분명히 적어야 합니다.

### 따라하기 안내: 데이터 출처 확인

**이 셀에서 하는 일**  
지금 읽은 데이터가 어디에서 왔는지 확인합니다.

**확인할 점**  
source path와 실행 범위를 봅니다.

**왜 중요한가**  
prepared artifact인지 local 재생성인지 구분해야 보고서가 정확해집니다.


### 출력 확인: `context.provenance`

**확인할 점**  
데이터나 artifact가 어디에서 왔는지 봅니다. 경로와 실행 범위를 먼저 확인합니다.

**왜 중요한가**  
같은 숫자라도 prepared artifact인지 local 재생성인지에 따라 보고서 표현이 달라집니다.

**다음 단계**  
이 범위를 기억한 상태로 다음 표를 읽습니다.


In [22]:
display(context.provenance)


,package_module,data_path,data_source,execution_scope,row_count,column_count
0,ai_quality.lite,data/vital_signs_evaluation_baseline.csv,embedded JupyterLite sample,embedded_fallback_sample,12,10


### 출력 확인: `context.preview`

**확인할 점**  
몇 개 행을 직접 보고 컬럼 이름과 값 모양이 예상과 맞는지 봅니다.

**왜 중요한가**  
Preview는 전체 판단이 아니라 데이터 모양 확인입니다.

**다음 단계**  
분포나 metric은 뒤의 요약 표에서 확인합니다.


In [23]:
display(context.preview)


,patient_id,timestamp,heart_rate,respiratory_rate,body_temperature,oxygen_saturation,systolic_blood_pressure,diastolic_blood_pressure,label
0,p-001,2026-01-01T09:00:00Z,72.0,16,36.6,98,118,76,low_risk
1,p-002,2026-01-01T09:01:00Z,118.0,25,37.9,92,146,92,high_risk
2,p-003,2026-01-01T09:02:00Z,63.0,15,36.4,99,110,70,low_risk
3,p-004,2026-01-01T09:03:00Z,132.0,28,38.4,89,158,96,high_risk
4,p-005,2026-01-01T09:04:00Z,84.0,18,36.8,97,126,78,low_risk


### 출력 확인: `context.label_distribution`

**확인할 점**  
`high_risk`와 `low_risk`의 개수와 비율을 봅니다.

**왜 중요한가**  
`high_risk`는 관심 class입니다. 이 수가 적으면 recall 같은 지표가 크게 흔들릴 수 있습니다.

**다음 단계**  
두 class가 모두 있는지 확인한 뒤 label gate로 넘어갑니다.


In [24]:
display(context.label_distribution)


,label,count,ratio_pct
0,low_risk,6,50.0
1,high_risk,6,50.0


### 3-2. QA 해석

이 출력에서 확인할 핵심은 `package_module`, `data_source`, `execution_scope`, 라벨 분포입니다. `package_module`이 `ai_quality.lite`이면 notebook이 `ai-quality` Lite 패키지 기준을 사용하고 있다는 뜻입니다. `data_source`가 `embedded JupyterLite sample`이면 실제 CSV를 읽지 못한 fallback 상태이므로 보고서에 fallback 기준이라고 적어야 합니다.

`high_risk`와 `low_risk`가 함께 존재하면 라벨 기준은 다음 gate에서 더 확인할 수 있습니다. 한쪽 라벨이 없거나 예상하지 못한 라벨이 보이면 모델 평가보다 라벨 생성 기준 확인을 먼저 요청해야 합니다.

## 4. Schema gate

### 4-1. 필수 컬럼과 타입 계약 확인

Schema gate는 모델 입력과 정답 비교가 가능한 구조인지 확인합니다. 필수 컬럼이 누락되면 모델 평가 결과를 계산하더라도 원인 해석이 흔들립니다. 이 단계에서 확인할 대상은 모델 입력 feature, 추적을 위한 `patient_id`와 `timestamp`, 그리고 정답 기준인 `label`입니다.

컬럼이 존재해도 타입이 어긋나면 품질 판단은 여전히 불안정합니다. 숫자형 feature가 문자열처럼 들어오면 범위 검증과 score 계산이 왜곡될 수 있고, `timestamp`가 parse되지 않으면 특정 시간대 또는 배치 문제를 추적하기 어렵습니다.

### 따라하기 안내: Schema gate

**이 셀에서 하는 일**  
필수 컬럼이 모두 있는지 확인합니다.

**확인할 점**  
모델 feature와 label 컬럼이 빠지지 않았는지 봅니다.

**왜 중요한가**  
컬럼이 빠지면 모델 평가를 진행할 수 없거나 결과 의미가 달라집니다.


In [25]:
schema_gate = ch01.build_schema_gate(dataframe)
schema_decision, missing_columns = ch01.summarize_schema_gate(schema_gate)


### 출력 확인: `schema_gate`

**확인할 점**  
필수 컬럼과 feature가 빠지지 않았는지 봅니다.

**왜 중요한가**  
필수 feature가 없으면 score 계산 조건이 달라지고, label이 없으면 metric을 계산할 수 없습니다.

**다음 단계**  
컬럼이 있으면 결측과 값 범위를 따로 확인합니다.

**주의할 점**  
컬럼 존재는 값 품질을 보장하지 않습니다.


In [26]:
display(schema_gate)


,column,role,required,exists,expected,observed
0,patient_id,traceability,True,True,identifier_or_supporting,object
1,heart_rate,feature,True,True,numeric,numeric_like
2,respiratory_rate,feature,True,True,numeric,numeric_like
3,body_temperature,feature,True,True,numeric,numeric_like
4,oxygen_saturation,feature,True,True,numeric,numeric_like
5,systolic_blood_pressure,feature,True,True,numeric,numeric_like
6,diastolic_blood_pressure,feature,True,True,numeric,numeric_like
7,timestamp,traceability,True,True,parseable_datetime,parseable
8,label,label,True,True,allowed_label,allowed_labels


### 출력 확인: `schema_decision`

**확인할 점**  
필수 컬럼과 feature가 빠지지 않았는지 봅니다.

**왜 중요한가**  
필수 feature가 없으면 score 계산 조건이 달라지고, label이 없으면 metric을 계산할 수 없습니다.

**다음 단계**  
컬럼이 있으면 결측과 값 범위를 따로 확인합니다.

**주의할 점**  
컬럼 존재는 값 품질을 보장하지 않습니다.


In [27]:
display(schema_decision)


,gate,status,problem_count,qa_judgment
0,schema_gate,pass,0,필수 컬럼과 기본 타입 계약을 만족합니다.


### 4-2. QA 해석

Schema gate가 `pass`이면 데이터 구조 문제 가능성은 낮아집니다. 다만 이 단계는 값의 품질을 보장하지 않습니다. 컬럼이 있어도 값이 비어 있거나 허용 범위를 벗어나면 모델 평가 전제가 여전히 흔들릴 수 있습니다.

보고서에는 “필수 컬럼 누락 없음”처럼 구조 증거를 짧게 남깁니다. 실패하면 모델 평가를 진행하지 말고 데이터 추출 또는 컬럼명 정리 owner에게 확인을 요청합니다.

## 5. Missing gate

### 5-1. 결측 위치와 라벨 영향 확인

Missing gate의 핵심은 결측 개수보다 결측 위치입니다. 모델 입력 특성 결측은 score 계산을 흔들 수 있고, `label` 결측은 metric 계산 자체를 제한합니다. 같은 결측률이라도 feature 결측과 label 결측의 품질 영향은 다릅니다.

결측이 특정 class에 집중되는지도 함께 봐야 합니다. `high_risk` 샘플에 결측이 몰리면 Recall 해석이 불안정해질 수 있고, `low_risk` 샘플에 결측이 몰리면 FP 해석이 흔들릴 수 있습니다.

### 따라하기 안내: Missing gate

**이 셀에서 하는 일**  
결측치가 있는지 확인합니다.

**확인할 점**  
어떤 컬럼에 결측이 있는지 봅니다.

**왜 중요한가**  
결측은 개수보다 위치가 중요합니다. label 결측과 feature 결측은 영향이 다릅니다.


In [28]:
missing_gate = ch01.build_missing_gate(dataframe)
missing_label_impact = ch01.build_missing_label_impact(dataframe)
(
    missing_decision,
    missing_problem_count,
    feature_missing_count,
    label_missing_count,
) = ch01.summarize_missing_gate(dataframe, missing_gate)


### 출력 확인: `missing_gate`

**확인할 점**  
결측치가 어느 컬럼에 있는지 봅니다.

**왜 중요한가**  
결측은 개수보다 위치가 중요합니다. label 결측과 feature 결측은 영향이 다릅니다.

**다음 단계**  
중요 feature나 label에 결측이 있으면 제한 사항으로 남깁니다.

**주의할 점**  
결측이 적어도 중요한 컬럼이면 문제가 될 수 있습니다.


In [29]:
display(missing_gate)


,column,role,missing_count,missing_ratio_pct,gate
1,heart_rate,feature,1,8.33,review
3,body_temperature,feature,0,0.00,pass
6,diastolic_blood_pressure,feature,0,0.00,pass
8,label,label,0,0.00,pass
4,oxygen_saturation,feature,0,0.00,pass
0,patient_id,traceability,0,0.00,pass
2,respiratory_rate,feature,0,0.00,pass
5,systolic_blood_pressure,feature,0,0.00,pass
7,timestamp,traceability,0,0.00,pass


### 출력 확인: `missing_label_impact`

**확인할 점**  
결측치가 어느 컬럼에 있는지 봅니다.

**왜 중요한가**  
결측은 개수보다 위치가 중요합니다. label 결측과 feature 결측은 영향이 다릅니다.

**다음 단계**  
중요 feature나 label에 결측이 있으면 제한 사항으로 남깁니다.

**주의할 점**  
결측이 적어도 중요한 컬럼이면 문제가 될 수 있습니다.


In [30]:
display(missing_label_impact)


,label,affected_rows,share_pct
0,high_risk,1,100.0


### 출력 확인: `missing_decision`

**확인할 점**  
결측치가 어느 컬럼에 있는지 봅니다.

**왜 중요한가**  
결측은 개수보다 위치가 중요합니다. label 결측과 feature 결측은 영향이 다릅니다.

**다음 단계**  
중요 feature나 label에 결측이 있으면 제한 사항으로 남깁니다.

**주의할 점**  
결측이 적어도 중요한 컬럼이면 문제가 될 수 있습니다.


In [31]:
display(missing_decision)


,gate,status,feature_missing_count,label_missing_count,qa_judgment
0,missing_gate,review,1,0,결측 위치와 라벨 집중도를 확인하고 평가 제외 또는 보정 기준을 정해야 합니다.


### 5-2. QA 해석

Missing gate가 `pass`이면 결측이 이번 `high_risk` 비율 변화의 직접 원인일 가능성은 낮아집니다. `review`나 `fail`이면 결측이 특정 입력 특성이나 특정 class에 집중되는지 보고서에 적고, 데이터 owner에게 제외 기준이나 보정 기준을 요청합니다.

보고 문장은 “결측 없음”으로 끝나면 부족합니다. “모델 입력 특성과 라벨 결측이 확인되지 않아 결측으로 인한 평가 전제 훼손 가능성은 낮습니다”처럼 품질 판단까지 붙여야 합니다.

## 6. Range gate

### 6-1. 허용 범위 초과와 입력 오류 후보 확인

Range gate는 값이 존재하더라도 입력 의미가 깨졌는지 확인합니다. 이 기준은 실제 의료 판단 기준이 아니라, 실습용 데이터에서 명백한 수집 오류, 단위 오류, 오류 코드 유입을 찾기 위한 검증 규칙입니다.

범위 초과가 있으면 값을 삭제하기 전에 소스 시스템, 단위 변환, 입력 schema를 확인해야 합니다. QA 담당자는 이상치를 제거하는 사람이 아니라, 어떤 기준으로 어떤 값을 의심했고 그 값이 평가 해석에 어떤 제한을 만드는지 기록하는 사람입니다.

### 따라하기 안내: Range gate

**이 셀에서 하는 일**  
값이 허용 범위 안에 있는지 확인합니다.

**확인할 점**  
범위를 벗어난 feature와 개수를 봅니다.

**왜 중요한가**  
범위 오류는 데이터 수집이나 단위 문제일 수 있고, score를 흔들 수 있습니다.


In [32]:
range_gate = ch01.build_range_gate(dataframe)
range_label_impact = ch01.build_range_label_impact(dataframe)
range_decision, invalid_feature_total, invalid_value_total = ch01.summarize_range_gate(range_gate)


### 출력 확인: `ch01.numeric_profile(dataframe)`

**확인할 점**  
허용 범위를 벗어난 값이 있는지 봅니다.

**왜 중요한가**  
범위 오류는 데이터 수집, 단위, 전처리 문제일 수 있고 score를 흔들 수 있습니다.

**다음 단계**  
범위 오류가 있는 feature를 score/metric 변화 후보로 연결합니다.

**주의할 점**  
이상값을 바로 제거한다고 결론 내리지 않습니다.


In [33]:
display(ch01.numeric_profile(dataframe))


,count,mean,std,min,25%,50%,75%,max
heart_rate,11.0,98.909091,28.395262,63.0,75.000,88.0,122.000,141.0
respiratory_rate,12.0,21.500000,5.744563,14.0,16.750,20.5,26.250,31.0
body_temperature,12.0,37.308333,0.868079,36.2,36.575,37.1,37.975,38.8
oxygen_saturation,12.0,94.166667,4.706540,86.0,90.750,94.0,98.250,101.0
systolic_blood_pressure,12.0,135.500000,18.052952,110.0,121.000,134.0,149.000,164.0
diastolic_blood_pressure,12.0,84.666667,10.516942,70.0,75.750,85.0,92.500,102.0
age,12.0,53.000000,16.481394,29.0,40.250,53.5,66.000,77.0


### 출력 확인: `range_gate`

**확인할 점**  
허용 범위를 벗어난 값이 있는지 봅니다.

**왜 중요한가**  
범위 오류는 데이터 수집, 단위, 전처리 문제일 수 있고 score를 흔들 수 있습니다.

**다음 단계**  
범위 오류가 있는 feature를 score/metric 변화 후보로 연결합니다.

**주의할 점**  
이상값을 바로 제거한다고 결론 내리지 않습니다.


In [34]:
display(range_gate)


,column,role,min_value,max_value,invalid_count,invalid_ratio_pct,gate
3,oxygen_saturation,feature,0.0,100.0,1,8.33,review
6,age,supporting,0.0,120.0,0,0.00,pass
2,body_temperature,feature,30.0,45.0,0,0.00,pass
5,diastolic_blood_pressure,feature,30.0,150.0,0,0.00,pass
0,heart_rate,feature,1.0,250.0,0,0.00,pass
8,height_m,supporting,0.5,2.5,0,0.00,not_in_dataset
1,respiratory_rate,feature,1.0,80.0,0,0.00,pass
4,systolic_blood_pressure,feature,50.0,250.0,0,0.00,pass
7,weight_kg,supporting,1.0,300.0,0,0.00,not_in_dataset


### 출력 확인: `range_label_impact`

**확인할 점**  
허용 범위를 벗어난 값이 있는지 봅니다.

**왜 중요한가**  
범위 오류는 데이터 수집, 단위, 전처리 문제일 수 있고 score를 흔들 수 있습니다.

**다음 단계**  
범위 오류가 있는 feature를 score/metric 변화 후보로 연결합니다.

**주의할 점**  
이상값을 바로 제거한다고 결론 내리지 않습니다.


In [35]:
display(range_label_impact)


,column,label,invalid_count
0,oxygen_saturation,low_risk,1


### 출력 확인: `range_decision`

**확인할 점**  
허용 범위를 벗어난 값이 있는지 봅니다.

**왜 중요한가**  
범위 오류는 데이터 수집, 단위, 전처리 문제일 수 있고 score를 흔들 수 있습니다.

**다음 단계**  
범위 오류가 있는 feature를 score/metric 변화 후보로 연결합니다.

**주의할 점**  
이상값을 바로 제거한다고 결론 내리지 않습니다.


In [36]:
display(range_decision)


,gate,status,invalid_feature_total,invalid_value_total,qa_judgment
0,range_gate,review,1,1,"허용 범위 초과 값이 있어 수집 단위, 파싱, 오류 코드 유입 확인이 필요합니다."


### 6-2. QA 해석

Range gate가 `pass`이면 명백한 입력 범위 오류 후보는 약해집니다. 다만 분포 변화나 운영 drift까지 부정하는 것은 아닙니다. 4장과 5장에서 운영 로그와 입력 분포를 다시 확인해야 합니다.

범위 초과가 있으면 보고서에는 컬럼명, 초과 건수, 영향을 받은 라벨을 함께 적습니다. 그래야 데이터 수집 owner가 어떤 입력 경로를 먼저 확인해야 하는지 판단할 수 있습니다.

## 7. Label gate

### 7-1. 정답 기준과 class support 확인

Label gate는 모델 지표의 분모를 확인하는 단계입니다. Precision과 Recall은 `label`을 기준으로 계산되므로, 라벨 값이 허용 목록과 다르거나 관심 class 표본 수가 너무 적으면 모델 평가 해석이 불안정합니다.

이 실습에서는 브라우저 sample에서도 판단할 수 있도록 최소 support 기준을 `30`으로 둡니다. 이 기준은 성능 보장을 의미하지 않고, 실습에서 class별 지표를 읽을 최소한의 표본이 있는지 확인하는 장치입니다.

### 따라하기 안내: Label gate

**이 셀에서 하는 일**  
label 값과 class 수를 확인합니다.

**확인할 점**  
invalid label, missing label, high/low risk 개수를 봅니다.

**왜 중요한가**  
모델 예측은 label과 비교됩니다. 정답 기준이 안정적이어야 metric이 의미 있습니다.


In [37]:
label_gate = ch01.build_label_gate(dataframe)
(
    label_decision,
    positive_support,
    negative_support,
    invalid_label_count,
    support_problem,
) = ch01.summarize_label_gate(label_gate)


### 출력 확인: `label_gate`

**확인할 점**  
invalid label, missing label, class별 표본 수를 봅니다.

**왜 중요한가**  
정답 label에 문제가 있으면 모델 예측을 제대로 채점할 수 없습니다.

**다음 단계**  
label 기준이 유지되면 feature와 metric 해석으로 넘어갑니다.

**주의할 점**  
label gate가 통과해도 feature 품질까지 통과한 것은 아닙니다.


In [38]:
display(label_gate)


,label,count,ratio_pct,allowed,meets_min_support
1,high_risk,6,50.0,True,False
0,low_risk,6,50.0,True,False


### 출력 확인: `label_decision`

**확인할 점**  
pass/fail, warning, note를 봅니다.

**왜 중요한가**  
이 출력은 다음 단계로 넘어갈 수 있는지 판단하는 checkpoint입니다.

**다음 단계**  
fail 또는 warning이면 어떤 기준 때문에 그런지 바로 앞 표와 연결합니다.

**주의할 점**  
pass라고 해서 모든 품질 문제가 사라진 것은 아닙니다.


In [39]:
display(label_decision)


,gate,status,positive_support,negative_support,invalid_label_count,qa_judgment
0,label_gate,fail,6,6,0,라벨 기준 또는 class support가 부족해 모델 지표 해석 전 재확인이 필요...


### 7-2. QA 해석

Label gate가 `pass`이면 `high_risk`와 `low_risk`를 기준으로 2장 모델 지표를 계산할 수 있습니다. 이 말은 모델이 좋다는 뜻이 아니라, 모델 지표를 해석할 정답 기준이 준비되었다는 뜻입니다.

허용되지 않은 라벨이 있거나 class support가 부족하면 모델 지표 해석을 보류합니다. 이 경우에는 라벨 생성 기준, 샘플링 기준, 평가 데이터 범위를 먼저 확인해야 합니다.

## 8. Evidence packet과 handoff

### 8-1. Gate 결과를 보고서 근거로 조립

마지막 단계는 앞의 gate 결과를 보고서에 들어갈 evidence packet으로 바꾸는 것입니다. 이 표는 2장 모델 평가 notebook으로 넘길 handoff이며, reviewer가 “무슨 근거로 모델 평가를 시작했는가”라고 물었을 때 답하는 자료입니다.

판단은 세 가지 중 하나로 정리합니다. 모든 gate가 통과하면 `ready_for_model_evaluation`, 검토 항목은 있지만 지표 계산 자체는 가능하면 `conditional_model_evaluation`, 필수 구조나 라벨 기준이 깨지면 `hold_for_data_recheck`입니다.

### 따라하기 안내: Evidence packet

**이 셀에서 하는 일**  
앞에서 확인한 결과를 한 묶음으로 정리합니다.

**확인할 점**  
각 gate 상태와 다음 장으로 넘길 내용을 봅니다.

**왜 중요한가**  
여러 표를 하나의 판단 흐름으로 묶어야 보고서에 쓸 수 있습니다.


In [40]:
(
    gate_summary,
    evidence_packet,
    chapter_01_handoff,
    report_sentence,
) = ch01.build_evidence_packet(
    context=context,
    schema_decision=schema_decision,
    missing_decision=missing_decision,
    range_decision=range_decision,
    label_decision=label_decision,
    missing_columns=missing_columns,
    missing_problem_count=missing_problem_count,
    feature_missing_count=feature_missing_count,
    label_missing_count=label_missing_count,
    invalid_feature_total=invalid_feature_total,
    invalid_value_total=invalid_value_total,
    positive_support=positive_support,
    negative_support=negative_support,
    invalid_label_count=invalid_label_count,
    support_problem=support_problem,
)


### 출력 확인: `gate_summary`

**확인할 점**  
pass/fail, warning, note를 봅니다.

**왜 중요한가**  
이 출력은 다음 단계로 넘어갈 수 있는지 판단하는 checkpoint입니다.

**다음 단계**  
fail 또는 warning이면 어떤 기준 때문에 그런지 바로 앞 표와 연결합니다.

**주의할 점**  
pass라고 해서 모든 품질 문제가 사라진 것은 아닙니다.


In [41]:
display(gate_summary)


,gate,status,key_observation,qa_judgment,owner,next_action
0,schema_gate,pass,"missing_columns=[], problem_count=0",필수 컬럼과 기본 타입 계약을 만족합니다.,QA,결측과 범위 gate 근거를 함께 첨부합니다.
1,missing_gate,review,"feature_missing=1, label_missing=0",결측 위치와 라벨 집중도를 확인하고 평가 제외 또는 보정 기준을 정해야 합니다.,Data Engineering,결측 위치와 라벨 집중도를 확인합니다.
2,range_gate,review,"invalid_feature_total=1, invalid_value_total=1","허용 범위 초과 값이 있어 수집 단위, 파싱, 오류 코드 유입 확인이 필요합니다.",Data Engineering,소스 시스템과 단위 변환을 확인합니다.
3,label_gate,fail,"high_risk=6, low_risk=6, invalid_label=0",라벨 기준 또는 class support가 부족해 모델 지표 해석 전 재확인이 필요...,Labeling/Data Engineering,라벨 생성 기준과 평가 범위를 확인합니다.


### 출력 확인: `evidence_packet`

**확인할 점**  
앞 단계 결과가 어떤 evidence 묶음으로 정리됐는지 봅니다.

**왜 중요한가**  
packet과 handoff는 다음 장이나 최종 보고서로 넘길 요약입니다.

**다음 단계**  
남은 질문을 다음 장에서 계속 확인합니다.

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다.


In [42]:
display(evidence_packet)


,evidence,observed,qa_judgment,owner,next_action
0,data_provenance,"embedded_fallback_sample, rows=12, columns=10",데이터 출처와 규모를 보고서에 남길 수 있습니다.,QA,2장 모델 평가에서 같은 데이터 범위를 명시합니다.
1,schema_gate,"missing_columns=[], problem_count=0",필수 컬럼과 기본 타입 계약을 만족합니다.,QA,결측과 범위 gate 근거를 함께 첨부합니다.
2,missing_gate,"feature_missing=1, label_missing=0",결측 위치와 라벨 집중도를 확인하고 평가 제외 또는 보정 기준을 정해야 합니다.,Data Engineering,결측 위치와 라벨 집중도를 확인합니다.
3,range_gate,"invalid_feature_total=1, invalid_value_total=1","허용 범위 초과 값이 있어 수집 단위, 파싱, 오류 코드 유입 확인이 필요합니다.",Data Engineering,소스 시스템과 단위 변환을 확인합니다.
4,label_gate,"high_risk=6, low_risk=6, invalid_label=0",라벨 기준 또는 class support가 부족해 모델 지표 해석 전 재확인이 필요...,Labeling/Data Engineering,라벨 생성 기준과 평가 범위를 확인합니다.


### 출력 확인: `chapter_01_handoff`

**확인할 점**  
앞 단계 결과가 어떤 evidence 묶음으로 정리됐는지 봅니다.

**왜 중요한가**  
packet과 handoff는 다음 장이나 최종 보고서로 넘길 요약입니다.

**다음 단계**  
남은 질문을 다음 장에서 계속 확인합니다.

**주의할 점**  
packet이 있다고 분석이 끝난 것은 아닙니다.


In [43]:
display(chapter_01_handoff)


,chapter,decision,decision_reason,open_candidates,next_chapter_question
0,01_data_quality,hold_for_data_recheck,필수 구조 또는 라벨 기준 문제가 있어 모델 지표 해석을 보류합니다.,"label_gate, missing_gate, range_gate","모델 지표 변화가 threshold, FP/FN, 데이터 조건 변화 중 어디와 연결..."


### 8-2. QA 해석

Evidence packet은 보고서의 원문 재료입니다. 수강생은 이 표에서 `observed`와 `qa_judgment`를 그대로 가져오되, 데이터 출처가 Lite sample인지 로컬 전체 데이터인지 반드시 구분해야 합니다.

`ready_for_model_evaluation`이면 2장으로 넘어가 모델 지표를 계산합니다. `conditional_model_evaluation`이면 제한 사항을 함께 붙여 계산합니다. `hold_for_data_recheck`이면 모델 지표보다 데이터 owner 확인이 먼저입니다.

## 9. 보고 문장 작성

### 9-1. 2장으로 넘길 문장 초안

마지막 셀은 evidence packet을 자연어 보고 문장으로 조립합니다. 이 문장은 최종 release 결론이 아니라 2장 모델 평가로 넘어가기 전의 데이터 전제 확인 문장입니다.

수강생은 숫자를 바꾸어 적기보다, 어떤 gate가 통과했고 어떤 불확실성이 남았는지를 분명히 써야 합니다. 특히 Lite sample으로 실행했다면 “Lite sample 기준”이라는 제한을 빼면 안 됩니다.

### 따라하기 안내: 보고 문장 작성

**이 셀에서 하는 일**  
지금까지 본 결과를 문장으로 바꿉니다.

**확인할 점**  
통과한 기준과 남은 제한 사항이 문장에 들어 있는지 봅니다.

**왜 중요한가**  
표와 숫자를 그대로 복사하지 말고 QA 판단 문장으로 정리해야 합니다.


In [44]:
print(report_sentence)

1장 데이터 품질 확인 결과, embedded_fallback_sample 기준 rows=12, high_risk support=6, low_risk support=6입니다. 필수 컬럼 누락은 0건, 모델 입력 결측은 1건, 라벨 결측은 0건, 허용 범위 초과는 1건입니다. 따라서 현재 판단은 hold_for_data_recheck이며, 근거는 필수 구조 또는 라벨 기준 문제가 있어 모델 지표 해석을 보류합니다.


### 9-2. 실패 시 확인 포인트

실행이 실패하면 먼저 `package_module`, `data_source`, helper import 상태를 확인합니다. JupyterLite에서 파일을 찾지 못해 embedded sample로 떨어졌다면, 보고서에는 prepared fallback sample 기준이라고 적고 데이터 파일 배포 상태를 확인해야 합니다.

출력이 예상과 다르면 결론을 서두르지 않습니다. `context.provenance`, `context.label_distribution`, `schema_gate`를 다시 보고 잘못된 파일, 라벨 표준화 누락, 필수 컬럼 누락 중 어디에 가까운지 분리합니다.

판단이 흔들릴 때는 approval이나 hold를 단정하지 말고 재확인 조건을 남깁니다. 예를 들어 “필수 컬럼은 확인했지만 `oxygen_saturation` 범위 초과가 있어 데이터 수집 owner 확인 전 모델 지표 판단을 조건부로 진행합니다”처럼 현재 증거와 다음 action을 함께 씁니다.